
Generates stratified annotation sheets for manual gold-standard evaluation of ArabPhon parser output. Samples 150 words from the training set (60 low / 70 medium / 20 high Darija difficulty) and 50 words from the test set (15 low / 20 medium / 15 high) for independent annotation by two annotators. Output: two CSV files (annotation_train.csv, annotation_test.csv) saved to Google Drive for inter-annotator agreement (IAA) computation and gold parser accuracy evaluation.

# Cell 1 — Mount Drive

In [ ]:

# Mount Google Drive to access the Tashkeela dataset stored in /Arabphon/TASHKEELA_DIR
from google.colab import drive
drive.mount('/content/drive')

TASHKEELA_DIR = '/content/drive/MyDrive/Arabphon/TASHKEELA_DIR'



# Cell 18 — Annotation Sheet Generation


In [ ]:
# Cell — Generate annotation CSVs (separate train and test sheets)

import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/Arabphon/train.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/Arabphon/test.csv')

def sample_by_difficulty(df, low, medium, high, seed=42):
    s_low  = df[df['difficulty']=='low'].sample(low, random_state=seed)
    s_med  = df[df['difficulty']=='medium'].sample(medium, random_state=seed)
    s_high = df[df['difficulty']=='high'].sample(high, random_state=seed)
    return pd.concat([s_low, s_med, s_high]).reset_index(drop=True)

# 150 from train: 60 low / 70 medium / 20 high
train_sample = sample_by_difficulty(train_df, 60, 70, 20)
train_sample['gold_phonemes_annotator1'] = ''
train_sample['gold_phonemes_annotator2'] = ''
train_sample['notes'] = ''
train_sample = train_sample[['difficulty','word','phonemes','gold_phonemes_annotator1','gold_phonemes_annotator2','notes']]

# 100 from test: 25 low / 50 medium / 25 high
test_sample = sample_by_difficulty(test_df, 25, 50, 25)
test_sample['gold_phonemes_annotator1'] = ''
test_sample['gold_phonemes_annotator2'] = ''
test_sample['notes'] = ''
test_sample = test_sample[['difficulty','word','phonemes','gold_phonemes_annotator1','gold_phonemes_annotator2','notes']]

train_out = '/content/drive/MyDrive/Arabphon/annotation_train_old.csv'
test_out  = '/content/drive/MyDrive/Arabphon/annotation_test_old.csv'

train_sample.to_csv(train_out, index=False, encoding='utf-8-sig')
test_sample.to_csv(test_out,   index=False, encoding='utf-8-sig')

print(f"Train annotation sheet: {len(train_sample)} words → {train_out}")
print(train_sample['difficulty'].value_counts())
print(f"\nTest annotation sheet: {len(test_sample)} words → {test_out}")
print(test_sample['difficulty'].value_counts())

# Cell 19 - Annotation analysis for test set




In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

for enc in ['utf-8-sig', 'latin-1', 'cp1256', 'iso-8859-6']:
    try:
        df = pd.read_csv('/content/drive/MyDrive/Arabphon/annotation_test.csv', encoding=enc)
        print(f"✓ {enc}")
        break
    except:
        print(f"✗ {enc}")

df.columns = df.columns.str.strip()
print(df.columns.tolist())

# IAA: exact sequence match between two annotators
df['agree'] = df['Annotator 1 (Gadi)'] == df['Annotator 2 (Sadouk)']
iaa_exact = df['agree'].mean()
print(f"IAA exact match: {iaa_exact*100:.1f}% ({df['agree'].sum()}/100)")

# Parser accuracy vs Sadouk gold
df['parser_correct'] = df['phonemes'] == df['Annotator 2 (Sadouk)']
parser_acc = df['parser_correct'].mean()
print(f"Parser accuracy vs gold: {parser_acc*100:.1f}% ({df['parser_correct'].sum()}/100)")

# By difficulty
for diff in ['low','medium','high']:
    g = df[df['difficulty']==diff]
    print(f"{diff}: IAA={g['agree'].mean()*100:.1f}% | Parser={g['parser_correct'].mean()*100:.1f}%")

In [ ]:
print("\n--- IAA Disagreements (2 rows) ---")
disagree = df[~df['agree']][['word','phonemes','Annotator 1 (Gadi)','Annotator 2 (Sadouk)','Note']]
print(disagree.to_string())

print("\n--- Parser Errors (10 rows) ---")
errors = df[~df['parser_correct']][['word','phonemes','Annotator 2 (Sadouk)','Note']]
print(errors.to_string())

# Cell 20 - Annotation analysis for training set

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

for enc in ['utf-8-sig', 'latin-1', 'cp1256']:
    try:
        df = pd.read_csv('/content/drive/MyDrive/Arabphon/annotation_train.csv', encoding=enc)
        print(f"✓ {enc}")
        break
    except:
        print(f"✗ {enc}")

df.columns = df.columns.str.strip()
print(df.columns.tolist())
print(f"Shape: {df.shape}")

# Drop rows where either annotator is empty
df = df[df['Annotator 1 (Gadi)'].notna() & df['Annotator 2 (Sadouk)'].notna()]
df = df[df['Annotator 1 (Gadi)'].str.strip() != '']
df = df[df['Annotator 2 (Sadouk)'].str.strip() != '']
print(f"Annotated rows: {len(df)}")

# IAA exact match
df['agree'] = df['Annotator 1 (Gadi)'].str.strip() == df['Annotator 2 (Sadouk)'].str.strip()
iaa_exact = df['agree'].mean()
print(f"\nIAA exact match: {iaa_exact:.1%}  ({df['agree'].sum()} / {len(df)})")

for diff in ['low', 'medium', 'high']:
    sub = df[df['difficulty'] == diff]
    if len(sub): print(f"  {diff}: {sub['agree'].mean():.1%}  (n={len(sub)})")

# Cohen's kappa (token-level)
ann1_tokens, ann2_tokens = [], []
for _, row in df.iterrows():
    a1 = str(row['Annotator 1 (Gadi)']).strip().split('-')
    a2 = str(row['Annotator 2 (Sadouk)']).strip().split('-')
    min_len = min(len(a1), len(a2))
    ann1_tokens.extend(a1[:min_len])
    ann2_tokens.extend(a2[:min_len])

kappa = cohen_kappa_score(ann1_tokens, ann2_tokens)
print(f"\nCohen's kappa (token-level): {kappa:.4f}")

# Parser accuracy vs gold
df['parser_correct'] = df['phonemes'].str.strip() == df['Annotator 2 (Sadouk)'].str.strip()
parser_acc = df['parser_correct'].mean()
print(f"\nParser accuracy vs gold: {parser_acc:.1%}  ({df['parser_correct'].sum()} / {len(df)})")

for diff in ['low', 'medium', 'high']:
    sub = df[df['difficulty'] == diff]
    if len(sub): print(f"  {diff}: {sub['parser_correct'].mean():.1%}  (n={len(sub)})")

# Errors
print("\n--- Parser Errors ---")
errors = df[~df['parser_correct']][['word','phonemes','Annotator 2 (Sadouk)','Note']]
print(errors.to_string())

print("\n--- IAA Disagreements ---")
disagree = df[~df['agree']][['word','Annotator 1 (Gadi)','Annotator 2 (Sadouk)','Note']]
print(disagree.to_string())